In [14]:
import os, rasterio, sys, rioxarray, pyflwdir, shutil, shapely
sys.path.append('backend/app/')
from rasterio.io import MemoryFile
from netCDF4 import Dataset
import geopandas as gpd, pandas as pd
import numpy as np, xarray as xr
from backend.app.services import flow_functions
from shapely.geometry import Polygon, MultiPolygon
from shapely import force_2d
from pyflwdir import dem
from scipy.ndimage import sobel
from hydromt_wflow import WflowSbmModel
from tqdm import tqdm
from owslib.wcs import WebCoverageService
from shapely.geometry import shape
from shapely.ops import unary_union
from rasterio.features import shapes
from whitebox.whitebox_tools import WhiteboxTools
wtb = WhiteboxTools()
wtb.set_verbose_mode(False)
np.random.seed(42)

## Child Functions

In [ ]:
def create_forcing(time, ny, nx, values, single_value=True):
    if single_value:
        data = np.empty((len(time), ny, nx), dtype=np.float32)
        data[:] = values[:, None, None]
    

    return data

# def assign_nodata(path, nodata):
#     with rasterio.open(path, "r+") as src:
#         arr = src.read(1)
#         arr = np.where(np.isnan(arr), nodata, arr)
#         src.write(arr, 1)
#         src.nodata = nodata

In [2]:
sample_folder, test_folder = 'inputs', 'test'
catchment_path = os.path.join(sample_folder, 'catchment.geojson')
terrain_path = os.path.join(sample_folder, 'dtm10.tif')
soil_path = os.path.join(sample_folder, 'soil.geojson')
land_path = os.path.join(sample_folder, 'land.geojson')
river_path = os.path.join(sample_folder, 'river.geojson')
terrain = rioxarray.open_rasterio(terrain_path).squeeze()
catchment = gpd.read_file(catchment_path)
initial_param = [0.25,0.3,50,0,0,500,0,0,100]
land = gpd.read_file(land_path)
river = gpd.read_file(river_path)

## Create merit hydro data

In [18]:
# Create raw terrain
raw_dir = os.path.normpath(os.path.join(f'{test_folder}/data/raw'))
if not os.path.exists(raw_dir): os.makedirs(raw_dir)
catchment_UTM = catchment.to_crs(terrain.rio.crs)
terrain_clipped = flow_functions.clip_catchment(catchment_UTM, terrain)
raw_path = os.path.normpath(os.path.join(raw_dir, "dtm_raw.tif"))
terrain_clipped.rio.to_raster(raw_path)

In [72]:
with rasterio.open(raw_path) as src:
    dem_array = src.read(1).astype(np.float32)
    profile, transform, crs = src.profile, src.transform, src.crs
hydro_dir = os.path.normpath(f'{test_folder}/data/hydro')
if os.path.exists(hydro_dir): shutil.rmtree(hydro_dir); os.makedirs(hydro_dir)
# Fill depressions
# fill_path = os.path.normpath(os.path.join(hydro_dir, "elevtn.tif"))
# wtb.fill_depressions(
#     dem=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\raw\test.tif",
#     output=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\elevtn.tif"
# )
# # Create flow direction
# flwdir_path = os.path.normpath(os.path.join(hydro_dir, "flwdir.tif"))
# wtb.d8_pointer(
#     dem=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\elevtn.tif", 
#     output=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\flwdir.tif"
# )
# # Create upstream area
# uparea_path = os.path.normpath(os.path.join(hydro_dir, "uparea.tif"))
# wtb.d8_flow_accumulation(
#     i=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\flwdir.tif", 
#     output=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\uparea.tif",
#     out_type="catchment area"
# )
# # Create basins
# wtb.basins(
#     d8_pntr=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\flwdir.tif",
#     output=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\basins.tif"
# )
# # Create stream order
# # strord_path = os.path.normpath(os.path.join(hydro_dir, "strord.tif"))
# wtb.extract_streams(
#     flow_accum=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\uparea.tif",
#     output=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\streams.tif",
#     threshold=100
# )
# wtb.strahler_stream_order(
#     d8_pntr=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\flwdir.tif",
#     streams=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\streams.tif",
#     output=r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\strord.tif"
# )






NODATA_DEM, NODATA_INT = -9999.0, 0
with rasterio.open(r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\raw\test.tif") as src:
    dem_array = src.read(1)
    transform, profile = src.transform, src.profile
    crs, nodata = src.crs,  src.nodata
# # Fill depressions
# filled_array, flwdir_array = dem.fill_depressions(elevtn=dem_array, max_depth=-1)
# # Create flow direction
flw = pyflwdir.from_dem(dem_array, transform=transform)
# # Create slope
# dx, dy = transform.a, abs(transform.e)
# # Gradient elevation
# gy, gx = np.gradient(filled_array, dy, dx)
# slope_array = np.arctan(np.sqrt(gx**2 + gy**2))
# slope_array = np.where(filled_array == NODATA_DEM, NODATA_DEM, slope_array).astype(np.float32)
# # # Create basins
# # basins_array = flw.basins()
# # Create stream order
# # uparea_array = flw.upstream_area(unit='km2')
# uparea_array = flw.accuflux(np.ones_like(filled_array))
# # # Create stream mask and stream order
# stream_mask = uparea_array > 0
# strord_array = flw.stream_order(type='strahler', mask=stream_mask)

# flow_functions.write_geotiff(filled_array, profile, os.path.join(hydro_dir, 'elevtn.tif'))
# profile_flwdir = {**profile, 'dtype': np.uint8, 'nodata': NODATA_INT}
# flow_functions.write_geotiff(flwdir_array, profile_flwdir, os.path.join(hydro_dir, 'flwdir.tif'))
# profile_slope = {**profile, 'dtype': np.float32, 'nodata': NODATA_DEM}
# flow_functions.write_geotiff(slope_array, profile_slope, os.path.join(hydro_dir, 'lndslp.tif'))
# # profile_basins = {**profile, 'dtype': np.int32, 'nodata': NODATA_INT}
# # flow_functions.write_geotiff(basins_array, profile_basins, os.path.join(hydro_dir, 'basins.tif'))
# profile_uparea = {**profile, 'dtype': np.float32, 'nodata': NODATA_DEM}
# flow_functions.write_geotiff(uparea_array, profile_uparea, os.path.join(hydro_dir, 'uparea.tif'))
# profile_strord = {**profile, 'dtype': np.int16, 'nodata': NODATA_INT}
# flow_functions.write_geotiff(strord_array, profile_strord, os.path.join(hydro_dir, 'strord.tif'))



# # Convert upstream area from m^2 to km^2
# nodata = -9999.0
# path = r"C:\Users\vanln\Downloads\Hydro-AI-Platform\test\data\hydro\uparea.tif"
# with rasterio.open(path) as src:
#     arr = src.read(1).astype("float32")
#     profile = src.profile.copy()
#     if src.nodata is not None: nodata = src.nodata
# mask = (arr == nodata)
# arr = arr / 1e6
# arr[mask] = nodata
# profile.update(dtype="float32", nodata=nodata)
# with rasterio.open(path, "w", **profile) as dst:
#     dst.write(arr, 1)

In [74]:
np.unique(flw.to_array(ftype='d8'))

array([  0,   1,   2,   4,   8,  16,  32,  64, 128, 247], dtype=uint8)

In [40]:
pyflwdir.from_array(flwdir_array, ftype='d8').to_array(ftype='d8')

array([[  2,   4,   8, ...,   2,   2,   4],
       [  2,   4,   8, ...,   2,   2,   4],
       [  1,   2,   2, ...,   2,   2,   4],
       ...,
       [  2,   4,   8, ..., 247, 247, 247],
       [  1,   2,   4, ..., 247, 247, 247],
       [128,   1,   0, ..., 247, 247, 247]],
      shape=(1920, 1680), dtype=uint8)

## Prepare forcing data from the customized area

In [ ]:
# Read weather data
weather_path = os.path.join(sample_folder, 'alesund_weather.csv')
weather = pd.read_csv(weather_path, parse_dates=['datetime'], index_col='datetime')
weather_new = weather.loc['2025-01-01 00:00:00':'2025-01-10 00:00:00']

In [ ]:
# Create forcing nc file
time, crs = weather_new.index.to_numpy(), terrain.rio.crs
if crs is None: raise ValueError("Terrain has no crs")
ny, nx = terrain.rio.height, terrain.rio.width
forcing = {
    'precip': ['precip_mm', '(mm/h)'], 'temp': ['temp_C', '(degC)'],
    'kin': ['shortwave_Wm2', '(W/m^2)'], 'kout': ['longwave_Wm2', '(W/m^2)'],
    'wind': ['wind_mps', '(m/s)'], 'press_msl': ['pressure', '(Pa)']
}
forcing_dir = os.path.join(test_folder, 'data/forcing')
if not os.path.exists(forcing_dir): os.makedirs(forcing_dir)
out_path, datasets = os.path.join(forcing_dir, "my_forcing.nc"), {}
for item, values in forcing.items():
    data = weather_new[values[0]].values
    data_3d = create_forcing(time, ny, nx, data)
    datasets[item] = (('time', 'y', 'x'), data_3d, {'units': values[1]})
ds_final = xr.Dataset(
    data_vars=datasets, coords={"time": time, "y": terrain.y, "x": terrain.x}
)
ds_final.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
ds_final.rio.write_crs(crs, inplace=True)
encoding = {
    var: {"zlib": True, "complevel": 4, "shuffle": True, "chunksizes": (1, 256, 256)}
    for var in ds_final.data_vars
}
ds_final.to_netcdf(out_path, engine='netcdf4', encoding=encoding)

## Process river data

In [ ]:
# Create a random value for each river
river_UTM = river.to_crs(terrain.rio.crs)
cols = {'width': (0.05, 2), 'depth': (1, 5), 'manning_n': (0.03, 0.06)}
river_cols = river_UTM.columns.drop('geometry', errors='ignore')
for col in river_cols:
    river_UTM[col] = pd.to_numeric(river_UTM[col], errors='coerce')
for col, (low, high) in cols.items():
    river_mask = river_UTM[col].isna() | (river_UTM[col] == 'None')
    river_UTM.loc[river_mask, col] = np.round(np.random.uniform(low, high, river_mask.sum()), 3)
river_UTM = river_UTM[river_UTM.is_valid].reset_index(drop=True)
# Process river
river_UTM["geometry"] = river_UTM.geometry.apply(lambda g: force_2d(g))
river_UTM = river_UTM.rename(columns={'width': 'rivwth', 'depth': 'rivdph'})
river_UTM = river_UTM[['rivwth', 'rivdph', 'manning_n', 'geometry']]

In [ ]:
# Write file
river_UTM.to_file(os.path.normpath(os.path.join(f'{test_folder}/data/river', 'river.gpkg')), driver='GPKG')

## Process soil data

In [ ]:
# Initialize variables
soil_types = {
    'clay': 'clyppt', 'sand': 'sndppt', 'silt': 'sltppt', 
    'bdod': 'bd', 'soc': 'oc', 'phh2o': 'ph'
}
depths = {
    '0-5cm_mean': 'sl1', '5-15cm_mean': 'sl2', '15-30cm_mean': 'sl3',
    '30-60cm_mean': 'sl4', '60-100cm_mean': 'sl5', '100-200cm_mean': 'sl6'
}
soil_dir = os.path.join(f'{test_folder}/data/soil')
if not os.path.exists(soil_dir): os.makedirs(soil_dir)
min_lon, min_lat, max_lon, max_lat = catchment.total_bounds
bbox = (float(min_lon), float(min_lat), float(max_lon), float(max_lat))

In [12]:
# Create soil thickness
with rasterio.open(terrain_path) as src:
    meta = src.meta.copy()
meta.update({"dtype": "float32", "nodata": -9999.0})
data = np.ones((meta["height"], meta["width"]), dtype="float32") * 100
data[data == meta["nodata"]] = 100
with rasterio.open(os.path.join(soil_dir, 'soilthickness.tif'), "w", **meta) as dst:
    dst.write(data, 1)

In [9]:
# Download soil data from ISRIC: https://files.isric.org/soilgrids/latest/data/
for item, name in tqdm(soil_types.items(), total=len(soil_types), desc='Downloading soil data'):
    wcs = WebCoverageService(f'https://maps.isric.org/mapserv?map=/map/{item}.map', version='1.0.0')
    for type, value in depths.items():
        idx = f'{item}_{type}'
        response = wcs.getCoverage(
            identifier=idx, crs='EPSG:4326', bbox=bbox,
            format='image/tiff', resx=0.0025, resy=0.0025
        )
        with MemoryFile(response.read()) as memfile:
            with memfile.open() as src:
                data = rioxarray.open_rasterio(src, masked=True)
                data_reprojected = data.rio.reproject_match(terrain)
            data_reprojected.rio.to_raster(os.path.join(soil_dir, f'{name}_{value}.tif'))

## Process land cover

In [12]:
dt = xr.open_dataset(r"test\data\forcing\2m_dewpoint_temperature_ERA5_2025_01.nc")
dt.close()

In [20]:
dt.close()


In [ ]:
# Fix invalid land polygon
land_UTM = land.to_crs(terrain.rio.crs)
land_cols = ['land', 'LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
land_UTM = flow_functions.fix_invalid_polygon(land_UTM, land_cols)
land_layers = ['LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
for value in land_layers:
    land_path = os.path.normpath(os.path.join('test/data/landuse', f'{value}.tif'))
    write_tif(land_path, terrain, land_UTM, value)

In [30]:
# Create raster and lookup table (used for calibration)
land_class = land.to_crs(terrain.rio.crs).copy()
land_class['class'] = None
columns, table = np.unique(land_class['land'].values), {}
for id, item in enumerate(columns):
    temp = land_class[land_class['land'] == item]
    table[item] = np.float32(temp.iloc[0][land_layers].values)
    land_class.loc[temp.index, 'class'] = id
land_class = land_class[['class', 'geometry']]
land_class_path = os.path.normpath(os.path.join('test/data/lookup', 'land_classes.tif'))
write_tif(land_class_path, terrain, land_class, 'class')
# Create lookup table
lookup = pd.DataFrame.from_dict(table, orient='index', columns=land_layers)
lookup.index.name = 'landcover'
lookup.reset_index(inplace=True)
lookup.insert(0, 'class_id', lookup.index)
# Save lookup table to csv
lookup_csv_path = os.path.normpath(os.path.join('test/data/lookup', 'lookup_land.csv'))
lookup.to_csv(lookup_csv_path, index=False)

In [6]:
# Run HydroMT
model_path = os.path.normpath(f'{test_folder}/model')
if os.path.exists(model_path): shutil.rmtree(model_path)
!hydromt build wflow_sbm "./test/model" -i "./test/build.yml" -d "./test/config.yml" -v

2026-05-20 13:31:50,656 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-20 13:31:50,705 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from ./test/config.yml
2026-05-20 13:31:50,728 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-05-20 13:31:50,728 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Envs\hyd_ai\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-05-20 13:31:50,739 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-05-20 13:31:50,739 - hydromt.hydromt_wflow.components.config - config - INFO - Reading default config file from C:/Envs/hyd_ai/Lib/site-packages/hydromt_wflow/data/wflow_sbm/wflow_sbm.toml.
2026-05-20 13:31:50,747 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-20 13:31:50,748 - hydromt.model.model - model - INFO - build: setup_config
2026-05-20 13:31:50,749 - hydromt.m